# Practice 2 · CNN Architecture and Operations

**In class** · Goal: run your first code on HaiMa and learn how to interrupt it.

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

## Question 1 · Receptive Fields and Local Connectivity

Based on regarding "Receptive Fields", the receptive field size for `L` successive convolution layers with kernel size `K` is calculated as `1 + L * (K - 1)`. Additionally, local connectivity dramatically reduces parameters compared to fully connected layers.

Please implement the following functions:
1. `calc_receptive_field(L, K)`: Calculate the receptive field size based on the given formula.
2. `calc_local_vs_fc_params(H, W, C_in, C_out, K)`: Calculate the number of parameters for a fully connected layer versus a locally connected/convolutional layer. 
   - For the fully connected layer, assume the input `(C_in, H, W)` is flattened and connected to an output of size `(C_out, H, W)` (i.e., the number of output neurons is `C_out * H * W`). Remember to include biases.
   - For the convolutional layer, use kernel size `K` and `C_out` filters. Remember to include biases.
   - Return a tuple `(fc_params, conv_params)`.


In [1]:

def calc_receptive_field(L, K):
    """Calculate receptive field size for L layers of KxK convolution."""
    # 🟢 EDIT: change this line (Use the given formula to calculate the receptive field size)
    return 1+ L * (K-1)

def calc_local_vs_fc_params(H, W, C_in, C_out, K):
    """Calculate parameters for FC layer vs Conv layer."""
    # 🟢 EDIT: change this line (Compute the parameter counts for both layers, remember the biases)
    input_neurons = C_in * H * W
    output_neurons = C_out * H * W
    fc_params = input_neurons * output_neurons + output_neurons
    conv_params = C_out * (C_in * K * K + 1)
    return fc_params, conv_params
    


## Question 2 · CNN Convolution Layer Calculation

A neuron connects to a local region (Receptive field) and uses shared connections (convolution). 
Implement the following functions to calculate the output dimensions and parameters of a convolutional layer:
1. `conv_output_size(H, W, K, S, P)`: Calculate the output height and width. Return a tuple `(H_out, W_out)`. Formula: `Output = (Input + 2*P - K) / S + 1` (using integer division).
2. `conv_params(C_in, C_out, K)`: Calculate the number of trainable parameters in the convolutional layer, taking into account the biases and weight sharing.


In [ ]:

def conv_output_size(H, W, K, S, P):
    """Calculate the output height and width of a convolution layer."""
    # 🟢 EDIT: change this line (Calculate the output height and width based on the input size, padding, kernel, and stride)
    H_out = (H + 2 * P - K) // S + 1
    W_out = (W + 2 * P - K) // S + 1
    return H_out, W_out

def conv_params(C_in, C_out, K):
    """Calculate the number of parameters in a convolution layer."""
    # 🟢 EDIT: change this line (Calculate the number of weights and biases separately)
    weights = C_out * C_in * K * K
    biases = C_out
    return weights + biases
    


## Question 3 · LeNet-5 Architecture Tracing

Referencing the "Classic Architecture: LeNet-5", the specific architecture is detailed in the table.  
Implement a pure Python/NumPy function `trace_lenet5(input_shape)` that calculates the exact output shape of each step in the LeNet-5 architecture.

The architecture is:
1. Input: `(1, 28, 28)`
2. Conv1: `C_out=20, K=5, P=2, S=1`
3. ReLU
4. MaxPool: `K=2, S=2`
5. Conv2: `C_out=50, K=5, P=2, S=1`
6. ReLU
7. MaxPool: `K=2, S=2`
8. Flatten
9. Linear1: `500` output features
10. ReLU
11. Linear2: `10` output features

The function should return a list of tuples representing the shape at each step: `[(label, C, H, W) or (label, features)]`.  
For example, the first few items should look like: `[('Input', 1, 28, 28), ('Conv1', 20, 28, 28), ...]`



In [ ]:

import numpy as np

def trace_lenet5(input_shape):
    """Trace the dimensions of LeNet-5 based on the slides."""
    C, H, W = input_shape
    shapes = [('Input', C, H, W)]

    # Conv1: C_out=20, K=5, P=2, S=1
    C = 20
    K, P, S = 5, 2, 1
    # 🟢 EDIT: change this line (Update H and W using the conv output formula)
    H = (H + 2 * P - K) // S + 1
    W = (W + 2 * P - K) // S + 1
    shapes.append(('Conv1', C, H, W))
    shapes.append(('ReLU1', C, H, W))

    # MaxPool1: K=2, S=2
    # 🟢 EDIT: change this line (Pooling with K=2, S=2 halves the spatial size)
    H = (H - 2) // 2 + 1
    W = (W - 2) // 2 + 1
    shapes.append(('Pool1', C, H, W))

    # Conv2: C_out=50, K=5, P=2, S=1
    C = 50
    K, P, S = 5, 2, 1
    # 🟢 EDIT: change this line (Update H and W using the conv output formula)
    H = (H + 2 * P - K) // S + 1
    W = (W + 2 * P - K) // S + 1
    shapes.append(('Conv2', C, H, W))
    shapes.append(('ReLU2', C, H, W))

    # MaxPool2: K=2, S=2
    # 🟢 EDIT: change this line (Pooling with K=2, S=2 halves the spatial size)
    H = (H - 2) // 2 + 1
    W = (W - 2) // 2 + 1
    shapes.append(('Pool2', C, H, W))

    # Flatten
    # 🟢 EDIT: change this line (Compute the total number of elements after flattening)
    flatten_dim = C * H * W
    shapes.append(('Flatten', flatten_dim))

    shapes.append(('Linear1', 500))
    shapes.append(('ReLU3', 500))
    shapes.append(('Linear2', 10))

    return shapes
    


## Question 4 · CNN Architecture Dimension Calculation

In model design, we utilize a pre-trained network as shown in the diagram. The notation `X@YxY` represents `X` feature map channels with a spatial resolution of `YxY`. The operation types and main parameters are annotated.

Given details:
- Input: `3@32x32`
- Conv1 + ReLU: kernel = 3x3, pad = 1, stride = 1. Output is `16@C1xC1`.
- Max-Pooling1: kernel = 2x2, stride = 2. Output is `16@P1xP1`.
- Conv2 + ReLU: kernel = 3x3, pad = C2, stride = 1. Output is `32@16x16`.
- Max-Pooling2: kernel = 2x2, stride = 2. Output is `32@P2xP2`.
- Conv3 + ReLU: kernel = 3x3, pad = 1, stride = 1. Output is `64@C3xC3`.
- FC4 + ReLU: 1024 neurons.
- FC5 + ReLU: F2 output neurons.

Implement the function `compute_network_dims()` to calculate and return the values of `C1`, `C2`, `C3`, `P1`, `P2`, and `F2` based on the given parameters. The handwritten letters dataset is used for fine-tuning.

Return your results as a dictionary with keys: `"C1"`, `"C2"`, `"C3"`, `"P1"`, `"P2"`, `"F2"`.


In [ ]:

def compute_network_dims():
    """
    Calculate the dimensions of the CNN architecture based on the provided image.
    Returns a dictionary containing the values of C1, C2, C3, P1, P2, and F2.
    """
    # Input size
    H_in = 32

    # 🟢 EDIT: change this line (Please write out the calculation process and state the final value)
    # Conv1: (32 + 2*1 - 3) // 1 + 1 = 32.
    C1 = (H_in + 2 * 1 - 3) // 1 + 1

    # Pool1: (32 - 2) // 2 + 1 = 16.
    P1 = (C1 - 2) // 2 + 1

    # Conv2 must output 16: 16 = P1 + 2*C2 - 3 + 1.
    # Thus C2 is padding (not a spatial dimension): C2 = 1.
    conv2_size = 16
    C2 = (conv2_size - P1 + 3 - 1) // 2

    # Pool2: (16 - 2) // 2 + 1 = 8.
    P2 = (conv2_size - 2) // 2 + 1

    # Conv3: (8 + 2*1 - 3) // 1 + 1 = 8.
    C3 = (P2 + 2 * 1 - 3) // 1 + 1

    # Assume 26 English letter classes, without separate upper/lowercase classes.
    F2 = 26
    return {"C1": C1, "C2": C2, "C3": C3, "P1": P1, "P2": P2, "F2": F2}
    